In [2]:
import sys
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.insert(0, os.getcwd())
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import joblib
from src.data_utils import load_config, load_raw_data, clean_data, save_processed
from src.features import encode_categoricals, scale_features, engineer_features, get_feature_matrix

config = load_config("config/config.yaml")
df_raw = load_raw_data(config)

Dataset cargado: 20718 filas, 27 columnas


In [3]:

df_clean = clean_data(df_raw, config)
print(f"\nShape final: {df_clean.shape}")

 Duplicados eliminados: 82
 Canciones virales (>= 500,000,000 streams): 1270

Shape final: (20063, 28)


c:\Users\juane\OneDrive\Escritorio\Machine-Learning\src\data_utils.py:37: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
c:\Users\juane\OneDrive\Escritorio\Machine-Learning\src\data_utils.py:42: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a co

In [4]:

df_feat = engineer_features(df_clean)
print("Features creadas:")
print([c for c in df_feat.columns if c not in df_raw.columns])

Features creadas:
['viral', 'engagement_rate', 'log_Views', 'log_Likes', 'log_Comments', 'log_Stream', 'popularity_score']


In [5]:
df_feat, encoders = encode_categoricals(df_feat, config["features_categoricas"])
joblib.dump(encoders, "../models/label_encoders.pkl")
print(" Encoders guardados")

FileNotFoundError: [Errno 2] No such file or directory: '../models/label_encoders.pkl'

In [6]:

X = get_feature_matrix(df_feat, config, use_log=True)
y_reg  = np.log1p(df_feat[config["target_regression"]])   # log-transform para regresión
y_clf  = df_feat[config["target_classification"]]

X_train, X_test, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=config["test_size"], random_state=config["random_state"])
_, _, y_train_c, y_test_c = train_test_split(
    X, y_clf, test_size=config["test_size"], random_state=config["random_state"])

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Balance viral en train: {y_train_c.mean():.2%}")

Train: (16050, 21) | Test: (4013, 21)
Balance viral en train: 6.41%


In [7]:

X_train_sc, X_test_sc, scaler = scale_features(X_train, X_test)

In [8]:

import os; os.makedirs("../data/processed", exist_ok=True)
joblib.dump((X_train, X_test, X_train_sc, X_test_sc,
             y_train_r, y_test_r, y_train_c, y_test_c,
             list(X.columns)), "../data/processed/splits.pkl")

save_processed(df_feat, config)
print("\nPreparación completada. Datos listos para modelado.")

 Datos guardados en data/processed/datos_limpios.csv

Preparación completada. Datos listos para modelado.
